# Tutorial 0 — A Guided Tour of the Hugging Face Ecosystem

## 0. Mental model: five libraries, one workflow

![The Hugging Face ecosystem: huggingface_hub stores and serves models, datasets and Spaces; datasets, transformers and evaluate each draw from it; accelerate is an optional helper under transformers controlling where computation runs; the three produce real data, model predictions and an evaluation score in turn](images/huggingface_hub.png)

`huggingface_hub` is where everything is stored and found. `datasets`, `transformers`, and `evaluate` each
take one job from there, and `accelerate` is an optional helper that changes only *where* computation runs,
not what it computes. Each item along the bottom sits under the library that produces it, and that row —
real data → model predictions → a score — is the workflow you build in section 5.

Rule of thumb for *which library owns which job*:

| Question | Library |
|---|---|
| "Which models/datasets exist, and what are their licenses?" | `huggingface_hub` |
| "How do I turn text into a model and back?" | `transformers` |
| "Where do I get labeled data to test or train on?" | `datasets` |
| "Is my model actually any good?" | `evaluate` |
| "How do I run this faster / on more hardware?" | `accelerate` |
| "How do I let other people try my model in a browser?" | Spaces |

## 1. Install the libraries

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub datasets evaluate sentencepiece ipywidgets


### Code walkthrough — installation

`!` runs a shell command instead of Python. `-q` keeps the output short (drop it when an install
misbehaves), and `-U` upgrades anything already present — convenient here, though production code pins
exact versions instead.

## 2. Search the Hub before you download anything

The Hub hosts models, datasets, and Spaces. You can query metadata (downloads, license, task, size)
without pulling any weights — useful for choosing between candidates quickly.

![The Hugging Face model browser at huggingface.co/models, with the Tasks filter, name filter, and sort control highlighted](images/huggingface_overview.png)

The web UI at [huggingface.co/models](https://huggingface.co/models) exposes exactly the same filters the
API call below uses: the **Tasks** sidebar is `pipeline_tag=`, **Filter by name** is `search=`, and the
**Sort** control is `sort=`. Browsing is good for getting a feel for what exists; the API call is what you
reach for when you want a reproducible shortlist inside a script.


In [ ]:
from huggingface_hub import list_models

candidates = list_models(
    pipeline_tag="text-classification",
    search="sentiment",
    sort="downloads",
    limit=5,
)

for m in candidates:
    print(f"{m.id:55s} downloads={m.downloads:>10,}  likes={m.likes}")


### Code walkthrough — `list_models(...)`

The call queries **metadata only** — no weights are downloaded.

- **`pipeline_tag="text-classification"`** restricts results to repos tagged for that task, keeping
  unrelated ones out.
- **`search="sentiment"`** is a text search over repo name and metadata, narrowing classification models
  toward sentiment ones.
- **`sort="downloads"`** and **`limit=5`** rank by download count and keep a five-row shortlist.

What comes back is an iterable of metadata objects, not models. Each `m` carries `m.id` (the `owner/name`
repo id), `m.downloads`, and `m.likes`.

The f-string does two things worth knowing: **`:55s`** pads the id into a 55-character column so rows line
up, and **`:>10,`** right-aligns the count in ten characters with thousands separators.

Popularity is a discovery signal, not evidence of fit. License, language, size, and the model card still
decide.

### Before you run

Two details in the output below are easy to misread.

`info.private` is a real boolean. `info.gated` is not: it is `False` on an open repository, but on a gated
one it is the *string* `"auto"` or `"manual"`, naming how access gets granted. So test it with
`if info.gated:` — `if info.gated is True:` would silently skip past every gated model on the Hub.

`info.siblings` lists every file in the repository, and the model card is one of them: `README.md` appears
in that list. The card is not separate metadata bolted onto the repo, it is an ordinary file you can
download and read like any other. That is what makes "read the card before trusting the model" something a
script can do, not just a human.

In [ ]:
from huggingface_hub import model_info

MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
info = model_info(MODEL_ID)

print("Model ID:       ", info.id)
print("Private:        ", info.private)
print("Gated:          ", info.gated)
print("Pipeline tag:   ", info.pipeline_tag)
print("Library:        ", info.library_name)
print("License tag:    ", [t for t in (info.tags or []) if t.startswith("license:")])
print("Downloads (30d):", f"{info.downloads:,}")
print("Likes:          ", info.likes)
print("Last modified:  ", info.last_modified)

# Parameter count is only reported for repos that ship .safetensors weights.
safetensors = getattr(info, "safetensors", None)
if safetensors is not None:
    print("Parameters:     ", f"{safetensors.total:,}")

print("Files:")
for s in info.siblings:
    print(" -", s.rfilename)


### Code walkthrough — `model_info(...)` and repository metadata

`model_info(MODEL_ID)` requests repository metadata. No weights are downloaded and no model is instantiated.

### The same repository, two views

Everything the cell printed is also visible by eye on the model's page:

![The model card page for distilbert-base-uncased-finetuned-sst-2-english on the Hugging Face Hub](images/distilbert_base.png)

| On the model page | In the `info` object |
|---|---|
| the title, `distilbert/distilbert-base-uncased-finetuned-sst-2-english` | `info.id` |
| the `Text Classification` tag | `info.pipeline_tag` |
| the `Transformers` tag | `info.library_name` |
| the `License: apache-2.0` tag | the `license:apache-2.0` entry in `info.tags` |
| **Downloads last month** | `info.downloads` (last 30 days, not a lifetime total) |
| the **Like** counter | `info.likes` |
| **Model size — 67M params** | `info.safetensors.total` |
| the **Files and versions** tab | `info.siblings`, each with `s.rfilename` |
| no padlock, no access form | `info.private`, `info.gated` |

The page is for browsing; `model_info()` is the same facts in a form a script can branch on — which is what
you want when filtering twenty candidates, or asserting a license before a training run.

The right-hand column repays a closer look, since that is where most of the deciding information sits:

![Annotated guide to the model page sidebar, numbering downloads and activity, model files and size, hosted inference, the inference widget, the model tree of adapters, finetunes and quantizations, the training datasets, and Spaces using the model](images/model_card_details.png)


### What is actually in the repository

`info.siblings` lists the repository's files, and the **Files and versions** tab is that same list in the
browser:

![Annotated guide to the Files and versions tab, showing config.json, four alternative weight formats of 268 MB each, tokenizer files, and the commit history](images/model_files.png)

Two things are worth taking from it. The repo carries **several alternative weight formats** — safetensors,
PyTorch `.bin`, Rust, TensorFlow — each 268 MB of the same 67M parameters, so the 1.34 GB total is not your
download size: `pipeline()` takes `config.json`, the tokenizer files, and *one* compatible weight file. And
every repo is a git repo, with branches and a commit history, so a checkpoint can be pinned to a revision
when reproducibility matters.

### What metadata cannot tell you

The left of that page — **Model Details**, **Uses**, **Risks, Limitations and Biases**, **Training** — is the
`README.md` sitting in `info.siblings`, and it is prose written by humans. The API tells you this checkpoint
is Apache-2.0, English, 67M parameters, and does text classification. Only the card tells you it was tuned on
SST-2, reaches 91.3 on that dev set, and carries documented bias caveats.

Metadata narrows the field; the card decides. Filter by API, then read the cards of the two or three
survivors.

## 3. `transformers`: the same four-step recipe, every time

Every `transformers` task follows the same shape, whatever the model or task:

```text
text  →  tokenizer  →  model  →  post-processing  →  usable output
```

`pipeline()` bundles all four steps. Below we run **three different tasks** across three different
architectures with the same call shape, to show how much the library standardises.

![How pipeline() works: model and tokenizer files download once from the Hub into a local cache, then raw text flows through tokenizer, model forward pass, logits, softmax and label mapping to a prediction](images/pipeline2.png)

Two things in that diagram routinely surprise people. **The download happens once**: the first
`pipeline(...)` call pulls model and tokenizer files into a local cache (`~/.cache/huggingface/hub/`, or
wherever `HF_HOME` points), and everything after reads from disk — which is why the first cell below is slow
and the rest are quick. On Colab that cache dies with the runtime. **Inference runs on your machine**:
`pipeline()` is not a web API call, so after the download the forward pass happens on your own CPU or GPU and
your text never leaves the machine — which matters as soon as the text is not yours to hand to a third party.

The bottom row also names what the sketch above calls "post-processing": the model emits raw **logits**,
softmax turns them into probabilities, and the label mapping turns the winning index into a name like
`POSITIVE`. That is where the `score` below comes from.

### 3.1 Sentiment analysis

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

for text in [
    "This course finally made transformers click for me.",
    "I've spent three hours debugging a shape mismatch and I regret everything.",
]:
    print(text, "->", sentiment(text))


### Code walkthrough — the sentiment pipeline

The first argument, **`"sentiment-analysis"`**, is the task: it selects the pipeline class and how output is
interpreted — here a label plus a confidence score. **`model=`** pins the exact checkpoint. Without it you
get whichever default your version of the library happens to choose, which is a poor foundation for a
notebook you want to re-run.

No tokenizer is passed, so `pipeline()` loads the one belonging to that checkpoint.

Calling `sentiment(text)` tokenizes, runs the forward pass, and maps the winning class to a readable label.
The loop reuses the already-loaded pipeline rather than reloading per iteration. Arguments you will meet
later include `device` and `batch_size`.

### 3.2 Translation

### Before you run

`Helsinki-NLP/opus-mt-en-fr` translates English to French, and nothing else. The language pair is not a
setting you pass at call time — it is baked into the weights, which were trained on English→French data.
Feed it an English sentence while hoping for German and you will still get French back.

To translate into a different language you swap the checkpoint: `opus-mt-en-de` for German, `opus-mt-en-es`
for Spanish, and so on across the several hundred pairs OPUS-MT publishes. Massively multilingual models
such as NLLB or M2M100 invert the trade: one set of weights handles many directions and you pass the target
language as an argument, at the cost of being far larger than a single-pair model.

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

result = translator("Hugging Face makes it easy to share and reuse machine learning models.")
print(result[0]["translation_text"])


### Code walkthrough — the translation pipeline

**`"translation"`** selects the sequence-to-sequence pipeline; the checkpoint supplies the language pair, as
described above.

The return value is a **list of dictionaries even for a single input**, so the text is reached as
`result[0]["translation_text"]`. That shape catches people out constantly, and it is consistent across most
pipelines.

The mechanism differs from classification: an encoder-decoder model *generates* its output token by token
rather than choosing among fixed labels.

### 3.3 Text generation

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen3-0.6B")

out = generator(
    "The three main things I want to remember about the Hugging Face ecosystem are:",
    max_new_tokens=60,
    do_sample=False,
)
print(out[0]["generated_text"])


### Code walkthrough — generation arguments

**`"text-generation"`** selects causal (autoregressive) generation, and `Qwen/Qwen3-0.6B` is small enough to
run comfortably in a tutorial.

- **`max_new_tokens=60`** caps the *new* tokens generated after the prompt. Prefer it to `max_length`, which
  counts prompt and continuation together and so shifts as your prompt grows.
- **`do_sample=False`** turns sampling off. With the default single beam this makes decoding greedy — the
  highest-scoring token at every step — which is what makes the output reproducible.

`out[0]["generated_text"]` holds the result, and for this pipeline it normally includes the prompt followed
by the continuation.

Set `do_sample=True` and the knobs that shape randomness — `temperature`, `top_p`, `top_k` — begin to
matter.

## 4. `datasets`: where the data comes from

Models are only half the story — you'll constantly need labeled data to evaluate or fine-tune on.
The `datasets` library gives you a uniform interface to over a million datasets on the Hub, with
memory-mapped, streaming-friendly loading so you don't need to fit everything in RAM.

![The Hugging Face dataset browser at huggingface.co/datasets, with filters for modality, size in rows, file format and type alongside a list of datasets showing row counts and downloads](images/datasets.png)

The filters here are the ones that decide whether data is usable before you download any of it: **Size** in
rows, **Format** (`parquet` and `arrow` load fastest), **Modalities**, and task tags. The **Viewer** badge is
the one to look for — it lets you page through real rows in the browser, which beats discovering that the
columns are wrong after a long download. `list_datasets()` runs the same search from Python, and you use it
in the mini-lab.


In [ ]:
from datasets import load_dataset

reviews = load_dataset("rotten_tomatoes", split="test[:50]")

print(reviews)
print()
print("Features:", reviews.features)
print()
print("First example:", reviews[0])


### Code walkthrough — `load_dataset(...)` and split slicing

**`split="test[:50]"`** is a split expression, and the grammar is worth memorising: `"train"` for the whole
split, `"train[:100]"` for the first hundred rows, `"train[:10%]"` or `"train[10%:20%]"` for percentage
slices.

A 50-row slice keeps the notebook fast while using exactly the API a full evaluation would use.

What you get back is a typed table rather than parsed text: `reviews.features` is the schema, `reviews[0]`
is one row as a dict, and printing the dataset shows its columns and row count.

### Before you run

The label column is a `ClassLabel` — not a bare integer, and not a string. That is the `datasets` library
holding onto two things at once: each row stores a compact integer, while the *schema* remembers what those
integers mean.

`.names` returns the meanings in order, and a name's position is its class ID. For this dataset that is
`['neg', 'pos']`, so `0` is negative and `1` is positive. Read the mapping off `.names` rather than assuming
it: the next sentiment dataset you load may well have ordered the two classes the other way round, and
nothing in the integers themselves would warn you.

In [ ]:
print(reviews.features["label"])
print("Class names:", reviews.features["label"].names)


In [ ]:
import pandas as pd

pd.DataFrame(reviews[:5])


### Code walkthrough — a slice into pandas

Slicing a `Dataset` returns a dict of columns, which is precisely what `pd.DataFrame` expects.

Five rows is for eyeballing. Converting a large dataset to pandas costs RAM and gives up the advantages of
the memory-mapped format.

## 5. Putting it together: score a real model on real data with `evaluate`

This is the workflow you'll use constantly: run a model over a dataset, then use `evaluate` to turn raw
predictions into a trustworthy number instead of eyeballing a handful of examples.

In [ ]:
predictions = sentiment([ex["text"] for ex in reviews])
pred_labels = [0 if p["label"] == "NEGATIVE" else 1 for p in predictions]
true_labels = reviews["label"]

print("First 5 predictions:", pred_labels[:5])
print("First 5 true labels:", true_labels[:5])


### Code walkthrough — batch inference and label alignment

Passing a **list of strings** asks the pipeline for batched inference instead of one call per row.

```python
pred_labels = [0 if p["label"] == "NEGATIVE" else 1 for p in predictions]
```

This is the line that matters. The model returns names (`"NEGATIVE"`, `"POSITIVE"`), the dataset stores
integers, and metrics compare integers. Get this mapping backwards and you produce a perfectly plausible
score that is exactly wrong — nothing downstream will warn you.

Indexing with a **column name**, `reviews["label"]`, returns the whole column. The `[:5]` prints are a
sanity check; the full lists are what gets scored.

### Before you run

The number printed below is not a general accuracy, because of how the slice was taken. `rotten_tomatoes`
stores its test split sorted by label, so `test[:50]` is **all positive reviews** — one class, no negatives.
What the score really measures is how often the model says POSITIVE on positive input, and a model that
answered POSITIVE unconditionally would score a perfect 1.0 on it.

That is worth seeing rather than hiding, because it is the most common way an evaluation quietly flatters a
model, and it is why Exercise 5.1 asks you to check the class balance. For a number you could actually
quote, shuffle before slicing:

```python
reviews = load_dataset("rotten_tomatoes", split="test").shuffle(seed=0).select(range(50))
```

On balanced data the interesting question is the in- vs out-of-distribution one. This checkpoint was
fine-tuned on SST-2, which is also movie-review sentiment, so the domain shift here is mild and the score
tends to land a little below its reported SST-2 figure — not the collapse you would see on clinical notes or
on tweets in another language.

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
result = accuracy_metric.compute(predictions=pred_labels, references=true_labels)

print("Accuracy on 50 rotten_tomatoes test examples:", result["accuracy"])


### Code walkthrough — loading and computing a metric

`evaluate.load("accuracy")` returns the metric implementation together with the input format it expects.

`.compute(predictions=..., references=...)` takes model outputs and gold labels — `references` is Evaluate's
term for ground truth throughout the library. Accuracy itself is just \(\#\{i:\hat y_i = y_i\}/N\), and the
result arrives as a dict, hence `result["accuracy"]`.

Use the keyword arguments rather than positional ones: silently swapping predictions and references changes
the meaning of any asymmetric metric.

## 6. Spaces: sharing a running demo, not just weights

A **Space** is a small hosted app — Gradio, Streamlit, or a static site — that wraps a model so anyone can
try it in a browser. A checkpoint on its own asks your visitor to have Python and `transformers` installed.
A Space asks them to click a link.

Browse them at [huggingface.co/spaces](https://huggingface.co/spaces) — around 1.5 million apps, grouped by
what they do:

![The Hugging Face Spaces directory at huggingface.co/spaces, titled The AI App Directory, showing task categories and a grid of featured Spaces of the week](images/spaces.png)

Open one and you are using the model, not reading about it. This is the first card above — a Gradio app
wrapped around a music-generation model:

![The YuE2-3B Music Generator Space running, with text boxes for style and lyrics, radio groups for symbolic planning and render quality, a seed field, and a Generate song button](images/music_generator.png)

Everything on that page is a Gradio component: text boxes for style and lyrics, radio groups, a number field
for the seed, a button, and the **Examples** table at the bottom that fills the form when clicked. Two facts
about the hosting show through as well. **Running on ZERO** means the Space borrows a shared GPU instead of
holding one, which is why `queue: 6/6` appears while other people's requests go first. And the **Files** tab
is the repository itself, so the source of any Space you find useful is there to read.

A minimal Gradio Space, in full:

```python
import gradio as gr
from transformers import pipeline

pipe = pipeline("sentiment-analysis")

def classify(text):
    return pipe(text)[0]

gr.Interface(fn=classify, inputs="text", outputs="json").launch()
```

`gr.Interface` wires a Python function to a UI: **`fn`** is the function to call, **`inputs="text"`** makes a
textbox whose contents are passed to it, and **`outputs="json"`** renders the returned dict as structured
output. `.launch()` starts the server — in a Space, the hosting environment runs that process for you.

Note that `pipeline("sentiment-analysis")` here has no `model=`, so it takes the library default. Fine for a
sketch; not for something other people depend on. A real Space pins the model and its dependencies, handles
errors, and picks its hardware.

You push `app.py` and a `requirements.txt` to a Space repo the same way you would push a model with
`huggingface_hub`, and the Hub builds and hosts it.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Hub** | Hugging Face's hosting service for models, datasets, and Spaces. |
| **Model card** | The `README.md` in a model repo — license, intended use, limitations, metrics. |
| **`pipeline()`** | High-level wrapper: tokenizer + model + post-processing in one call, for one task. |
| **`datasets.Dataset`** | A memory-mapped table of examples with typed `features` (e.g. `ClassLabel`). |
| **`evaluate`** | A library of standard metrics (accuracy, F1, BLEU, ROUGE, ...) with a uniform `.compute()` API. |
| **`accelerate`** | Runs the same training/inference code on CPU, single GPU, multi-GPU, or TPU without rewrites. |
| **Space** | A hosted demo app (often Gradio) that wraps a model behind a browser UI. |
| **Gated model** | Requires accepting a license on the model page before you can download weights. |
| **In- vs out-of-distribution evaluation** | Testing on data similar to training data vs. testing on data that differs — scores can drop a lot on the latter. |

---
